### Install Dependencies (run once)

In [ ]:
!pip install matplotlib pandas pyxdf numpy mne

### Import Packages

In [ ]:
import mne
import pyxdf
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from matplotlib.collections import LineCollection, PolyCollection, PatchCollection
import matplotlib.patches as patches

### Data Loading

In [ ]:
xdf_path = "/Users/nadiapl/pupil-labs/LSL/eeg-neon-v1a.xdf"
streams, header = pyxdf.load_xdf(xdf_path)

def get_st(n): return next(s for s in streams if s['info']['name'][0] == n)

eeg_st = get_st("Explore_AABC_ExG")
gaze_st = get_st("Neon Companion_Neon Gaze")
marker_st = get_st("PSY_MARKERS_TASK")

# Define experiment window from markers
marker_ts = marker_st['time_stamps']
marker_names = [json.loads(m[0])['name'] if '{' in m[0] else m[0] for m in marker_st['time_series']]

t_start = marker_ts[marker_names.index("experiment_start")]
t_end = marker_ts[len(marker_names) - 1 - marker_names[::-1].index("experiment_end")]

# Filter EEG
eeg_mask = (eeg_st['time_stamps'] >= t_start) & (eeg_st['time_stamps'] <= t_end)
eeg_ts_w = eeg_st['time_stamps'][eeg_mask]
eeg_data_w = eeg_st['time_series'][eeg_mask]

# Resample Eye-Tracking to EEG clock
def resample(t_old, d_old, t_new):
    return interp1d(t_old, d_old, bounds_error=False, fill_value="extrapolate")(t_new)

g_ts, g_data = gaze_st['time_stamps'], gaze_st['time_series']
p_l = resample(g_ts, g_data[:, 2], eeg_ts_w)
p_r = resample(g_ts, g_data[:, 9], eeg_ts_w)

### MNE Object Creation

In [ ]:
eeg_ch_names = ["Fz", "FCz", "Cz", "CPz"]
ch_names = eeg_ch_names + ["Pupil Left", "Pupil Right"]
ch_types = (['eeg'] * len(eeg_ch_names)) + (['pupil'] * 2)

info = mne.create_info(ch_names, sfreq=float(eeg_st['info']['nominal_srate'][0]), ch_types=ch_types)
raw = mne.io.RawArray(np.vstack([eeg_data_w[:, :4].T * 1e-6, p_l, p_r]), info)

# --- Markers as Annotations ---
onsets = marker_ts[(marker_ts >= t_start) & (marker_ts <= t_end)] - t_start
labels = [n for t, n in zip(marker_ts, marker_names) if t_start <= t <= t_end]
raw.set_annotations(mne.Annotations(onset=onsets, duration=[0]*len(onsets), description=labels))
annotations = raw.annotations
# Keep only experiment-relevant events
mask = [desc != "condition_start" for desc in annotations.description]
new_annotations = annotations[mask]
# Apply the filtered list back to the raw object
raw.set_annotations(new_annotations)

### EEG Filtering (optional)

In [ ]:
#raw.notch_filter(50, picks='eeg', verbose=False) 
raw.filter(l_freq=1.0, h_freq=30.0, picks='eeg', verbose=False)

### Plotting

In [ ]:
scalings = dict(eeg=100e-6, pupil=0.5)
my_colors = dict(eeg="#305882", pupil="#B07B92")
plt.rcParams['figure.figsize'] = [30, 30] # Huge canvas
plt.rcParams['figure.dpi'] = 100          # High resolution
fig = raw.plot(
    duration=40, 
    n_channels=len(ch_names),
    scalings=scalings,
    color=my_colors,
    show_scrollbars=False,
    show_scalebars=False,
    remove_dc=True,
    show=False,
    clipping=None,
    title="EEG + Pupil Diameter"
)
# --- AXIS ADJUSTMENT ---
for i, ax in enumerate(fig.axes):
    labels = [t.get_text() for t in ax.get_yticklabels()]
    if any("Pupil" in label for label in labels) or i >= len(eeg_ch_names):
        curr_ymin, curr_ymax = ax.get_ylim()
        ax.set_ylim(curr_ymin * 1.1, curr_ymax * 1.1) 
        ax.set_zorder(2) # Ensure they stay on top

# Define our match color
MATCH_COLOR = "#706F78"
for ax in fig.axes:
    # 1. Style the Labels (Text)
    texts = [child for child in ax.get_children() if isinstance(child, plt.Text)]
    for i, txt in enumerate(texts):
        if txt.get_text().strip():
            txt.set_rotation(45)
            txt.fontsize = 9
            txt.set_color(MATCH_COLOR)
            txt.set_y([1.02, 1.10, 1.18][i % 3])
            txt.set_clip_on(False)
            txt.set_bbox(dict(facecolor='white', alpha=0.9, edgecolor='none', pad=2))

    for child in ax.get_children():
        if isinstance(child, (LineCollection, PolyCollection, PatchCollection, patches.Polygon)):
            child.set_edgecolor(MATCH_COLOR)
            child.set_facecolor('none')      # Ensure it's just a line, not a filled box
            child.set_linewidth(1)
            child.set_alpha(0.3)
            child.set_linestyle((0, (2, 2))) # Dashed
            child.set_clip_on(False)
            child.set_zorder(0)

        # Catch-all for any individual vertical lines that escaped
        elif isinstance(child, plt.Line2D):
            xdata = child.get_xdata()
            if len(np.unique(xdata)) == 1: # It's a vertical marker
                child.set_color(MATCH_COLOR)
                child.set_linestyle('--')
                child.set_linewidth(1.5)
                child.set_alpha(0.5)
                child.set_clip_on(False)    
    if ax.get_xlabel():
        ax.set_xlabel("Time (LSL seconds)", fontsize=10)

# Ensure the x-axis limits strictly match your experiment window
plt.xlim(t_start, t_end)
# Final formatting
plt.subplots_adjust(top=0.9, bottom=0.2, right=0.9)
plt.savefig("synced_event_plot.png", dpi=300, bbox_inches='tight')
plt.show()